<a href="https://colab.research.google.com/github/Tamannapanwar17/Artificial-Neural-Network/blob/main/minor_Basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q tenseal tensorflow scikit-learn matplotlib pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 59.8 MB/s eta 0:00:00


In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
import tenseal as ts
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Lambda
)

In [ ]:
print("TensorFlow:", tf.__version__)
print("TenSEAL:", ts.__version__)
print("NumPy:", np.__version__)

TensorFlow: 2.20.0
TenSEAL: 0.3.18
NumPy: 2.1.3


In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"tamannapanwar","key":"513b7290bd35147190485641dc827d96"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d paultimothymooney/breast-histopathology-images

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/breast-histopathology-images
License(s): CC0-1.0
100% 3.10G/3.10G [01:18<00:00, 42.4MB/s]



In [ ]:
!unzip -q breast-histopathology-images.zip -d /content/breast_histopathology

In [ ]:
import os
import glob
import shutil
import random

random.seed(42)

SOURCE = "/content/breast_histopathology/IDC_regular_ps50_idx5"
DEST = "/content/ckks_dataset"

# Create folders
for split in ["train", "validation", "test"]:
    for label in ["0", "1"]:
        os.makedirs(
            os.path.join(DEST, split, label),
            exist_ok=True
        )

# Collect images from class 0 and class 1
class_0 = glob.glob(os.path.join(SOURCE, "**", "0", "*.png"), recursive=True)
class_1 = glob.glob(os.path.join(SOURCE, "**", "1", "*.png"), recursive=True)

print("Class 0 images:", len(class_0))
print("Class 1 images:", len(class_1))

Class 0 images: 198738
Class 1 images: 78786


In [ ]:
random.shuffle(class_0)
random.shuffle(class_1)

class_0 = class_0[:100]
class_1 = class_1[:100]

print("Selected class 0:", len(class_0))
print("Selected class 1:", len(class_1))

Selected class 0: 100
Selected class 1: 100


In [ ]:
def split_images(images):
    train = images[:80]
    validation = images[80:90]
    test = images[90:100]

    return train, validation, test

train_0, val_0, test_0 = split_images(class_0)
train_1, val_1, test_1 = split_images(class_1)

In [ ]:
def copy_images(images, destination):
    for i, image_path in enumerate(images):
        filename = f"{i:04d}.png"
        shutil.copy(
            image_path,
            os.path.join(destination, filename)
        )

# Class 0
copy_images(train_0, f"{DEST}/train/0")
copy_images(val_0, f"{DEST}/validation/0")
copy_images(test_0, f"{DEST}/test/0")

# Class 1
copy_images(train_1, f"{DEST}/train/1")
copy_images(val_1, f"{DEST}/validation/1")
copy_images(test_1, f"{DEST}/test/1")

In [ ]:
# final dataset
for split in ["train", "validation", "test"]:
    for label in ["0", "1"]:
        path = os.path.join(DEST, split, label)
        count = len(os.listdir(path))
        print(f"{split:10s} class {label}: {count}")

train      class 0: 80
train      class 1: 80
validation class 0: 10
validation class 1: 10
test       class 0: 10
test       class 1: 10


In [ ]:
print(tf.config.list_physical_devices('GPU'))

[]


In [ ]:
IMG_SIZE = (50, 50)
BATCH_SIZE = 16

train_dataset = tf.keras.utils.image_dataset_from_directory(
    "/content/ckks_dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=42
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    "/content/ckks_dataset/validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    "/content/ckks_dataset/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

Found 160 files belonging to 2 classes.
Found 20 files belonging to 2 classes.
Found 20 files belonging to 2 classes.


In [ ]:
print(train_dataset.class_names)

['0', '1']


In [ ]:
for images, labels in train_dataset.take(1):
    print(images.shape)
    print(labels.shape)

(16, 50, 50, 3)
(16, 1)


# Build cnn model

In [ ]:
# 50×50×3
#    ↓
# Conv2D (kernel = 3, padding = same,16)
#    ↓
# ReLU
#    ↓
# MaxPooling (2X2)
#    ↓
# Conv2D (32)
#    ↓
# ReLU remaining all same
#    ↓
# MaxPooling
#    ↓
# Flatten
#    ↓
# Dense (64, relu) hidden -> (1,sigmoid)
#    ↓
# Output

In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
from tensorflow.keras import Sequential, layers

model = Sequential([

    Input(shape=(50, 50, 3)),

    # Normalize pixel values from [0, 255] to [0, 1]
    Rescaling(1.0 / 255),

    # CNN layer 1
    Conv2D(
        filters=16,
        kernel_size=(3, 3),
        padding="same",
        activation="relu"
    ),

    MaxPooling2D(
        pool_size=(2, 2)
    ),

    # CNN layer 2
    Conv2D(
        filters=32,
        kernel_size=(3, 3),
        padding="same",
        activation="relu"
    ),

    MaxPooling2D(
        pool_size=(2, 2)
    ),

    # Convert feature maps into a vector
    Flatten(),

    # Fully connected layer
    Dense(
        64,
        activation="relu"
    ),

    # Binary classification
    Dense(
        1,
        activation="sigmoid"
    )
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 50, 50, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 50, 50, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 25, 25, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 25, 25, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       294,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 300,129 (1.14 MB)

 Trainable params: 300,129 (1.14 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow import keras
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10
)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.8375 - loss: 0.3591 - val_accuracy: 0.5000 - val_loss: 0.7949
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.7063 - loss: 0.5201 - val_accuracy: 0.5500 - val_loss: 0.7318
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.7937 - loss: 0.4458 - val_accuracy: 0.6000 - val_loss: 0.6228
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - accuracy: 0.7937 - loss: 0.4381 - val_accuracy: 0.7000 - val_loss: 0.6166
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 86ms/step - accuracy: 0.8500 - loss: 0.3650 - val_accuracy: 0.7500 - val_loss: 0.6172
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - accuracy: 0.8750 - loss: 0.3443 - val_accuracy: 0.7000 - val_loss: 0.5739
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 86ms/step - accuracy: 0.8562 - loss: 0.3566 - val_accuracy: 0.5000 - val_loss: 0.8165
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.8313 - loss: 0.3746 - val_accuracy: 0.6000 - v

In [ ]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6000 - loss: 0.5715
Test loss: 0.5714801549911499
Test accuracy: 0.6000000238418579


In [ ]:
images, labels = next(iter(test_dataset))

image = images[0]
true_label = labels[0].numpy()[0]

print("Image shape:", image.shape)
print("True label:", true_label)

Image shape: (50, 50, 3)
True label: 0.0


In [ ]:
image_batch = tf.expand_dims(image, axis=0)

In [ ]:
prediction = model.predict(image_batch)

print("Prediction probability:", prediction[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
Prediction probability: 0.20146516


In [ ]:
predicted_class = 1 if prediction[0][0] >= 0.5 else 0

print("Predicted class:", predicted_class)
print("True class:", int(true_label))

Predicted class: 0
True class: 0


In [ ]:
model.save("/content/cnn_model.keras")

In [ ]:
print(os.path.exists("/content/cnn_model.keras"))

True


In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("/content/cnn_model.keras")

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 50, 50, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 50, 50, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 25, 25, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 25, 25, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       294,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 900,389 (3.43 MB)

 Trainable params: 300,129 (1.14 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 600,260 (2.29 MB)

In [ ]:
import tenseal as ts

In [ ]:
# This creates the cryptographic environment required for CKKS operations.
context = ts.context(
    ts.SCHEME_TYPE.CKKS, # This tells TenSEAL to use the CKKS homomorphic encryption scheme.
    poly_modulus_degree=16384,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)

context.global_scale = 2**40 # we're choosing approximately 40 bits of scale precision for the CKKS calculations.

In [ ]:
# Basic operations
x = [1.5, 2.5, 3.5]

encrypted_x = ts.ckks_vector(context, x)

encrypted_result = encrypted_x * 2

decrypted_result = encrypted_result.decrypt()

print(decrypted_result)

[3.000004291598442, 5.000007152229254, 7.000010009168978]


In [ ]:
# Take one test image
images, labels = next(iter(test_dataset))

image = images[0]
label = labels[0].numpy()[0]

print("Image shape:", image.shape)
print("True label:", int(label))

Image shape: (50, 50, 3)
True label: 0


In [ ]:
# CKKS works with vectors of numerical values, so flatten it:
image_vector = tf.reshape(image, [-1]).numpy()

print("Vector shape:", image_vector.shape)
print("Number of values:", len(image_vector))
print("First 10 values:", image_vector[:10])

Vector shape: (7500,)
Number of values: 7500
First 10 values: [237. 227. 232. 211. 154. 184. 210. 140. 172. 188.]


In [ ]:
image_vector = image_vector / 255.0

print("First 10 normalized values:")
print(image_vector[:10])

First 10 normalized values:
[0.92941177 0.8901961  0.9098039  0.827451   0.6039216  0.72156864
 0.8235294  0.54901963 0.6745098  0.7372549 ]


In [ ]:
# This creates the cryptographic environment required for CKKS operations.
context.generate_galois_keys()

In [ ]:
encrypted_image = ts.ckks_vector(
    context,
    image_vector.tolist()
)

print("Image encrypted successfully.")

Image encrypted successfully.


In [ ]:
decrypted_image = encrypted_image.decrypt()

print("First 10 original values:")
print(image_vector[:10])

print("\nFirst 10 decrypted values:")
print(decrypted_image[:10])

First 10 original values:
[0.92941177 0.8901961  0.9098039  0.827451   0.6039216  0.72156864
 0.8235294  0.54901963 0.6745098  0.7372549 ]

First 10 decrypted values:
[0.9294117705017273, 0.8901960846241935, 0.9098039277374479, 0.8274509912338884, 0.6039215927273189, 0.7215686427880615, 0.8235294222071445, 0.5490196340951504, 0.6745098235581966, 0.737254919069543]


In [ ]:
# Get the Dense layer weights
for layer in model.layers:
    print(layer.name, layer.__class__.__name__)

rescaling_1 Rescaling
conv2d_2 Conv2D
max_pooling2d_2 MaxPooling2D
conv2d_3 Conv2D
max_pooling2d_3 MaxPooling2D
flatten_1 Flatten
dense_2 Dense
dense_3 Dense


In [ ]:
# Now extract the first Dense layer:
dense_layer = model.get_layer("dense_2")

weights, bias = dense_layer.get_weights()

print("Weights shape:", weights.shape)
print("Bias shape:", bias.shape)

Weights shape: (4608, 64)
Bias shape: (64,)


In [ ]:
# Take one image
images, labels = next(iter(test_dataset))
image = images[0]

# Normalize
image = image / 255.0

# Flatten
x = tf.reshape(image, [-1]).numpy()

print("Input size:", len(x))

Input size: 7500


For CKKS, we will initially use a simpler architecture based mainly on matrix multiplication + addition + polynomial activation.

In [ ]:
import tensorflow as tf
import numpy as np
import random
import os

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ["TF_DETERMINISTIC_OPS"] = "1"

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

ckks_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(50, 50, 3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32),
    tf.keras.layers.Lambda(lambda x: x * x),
    tf.keras.layers.Dense(16),
    tf.keras.layers.Lambda(lambda x: x * x),
    tf.keras.layers.Dense(1)
])

ckks_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [ ]:
history = ckks_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=100
)
# 10 - 0.5
# 300 - 120.578857421875
# Test accuracy: 0.699999988079071

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.4875 - loss: 1.5089 - val_accuracy: 0.5000 - val_loss: 1.3322
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4625 - loss: 0.9866 - val_accuracy: 0.5000 - val_loss: 0.6094
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4875 - loss: 0.7354 - val_accuracy: 0.5500 - val_loss: 0.6240
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4750 - loss: 0.6727 - val_accuracy: 0.5000 - val_loss: 0.7129
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.5063 - loss: 0.6647 - val_accuracy: 0.5000 - val_loss: 0.7328
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5063 - loss: 0.6372 - val_accuracy: 0.5000 - val_loss: 0.6251
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5000 - loss: 0.6138 - val_accuracy: 0.5000 - val_loss: 0.7443
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.5125 - loss: 0.6491 - val_accuracy: 0.

In [ ]:
test_loss, test_accuracy = ckks_model.evaluate(test_dataset)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.7000 - loss: 0.7788
Test loss: 0.778837263584137
Test accuracy: 0.699999988079071


In [ ]:
ckks_model.save("/content/ckks_friendly_model.keras")

In [ ]:
for i, layer in enumerate(ckks_model.layers):
    print(i, layer.name, layer.__class__.__name__)

0 rescaling_2 Rescaling
1 flatten_2 Flatten
2 dense_4 Dense
3 lambda Lambda
4 dense_5 Dense
5 lambda_1 Lambda
6 dense_6 Dense


In [ ]:
dense1 = ckks_model.layers[2]

weights1, bias1 = dense1.get_weights()

print("Weights shape:", weights1.shape)
print("Bias shape:", bias1.shape)

Weights shape: (7500, 32)
Bias shape: (32,)


In [ ]:
images, labels = next(iter(test_dataset))

image = images[0]

x = image.numpy() / 255.0
x = x.reshape(-1)

print("Input shape:", x.shape)

Input shape: (7500,)


In [ ]:
plain_output = x @ weights1 + bias1

print("Plain output shape:", plain_output.shape)
print("First 5 outputs:", plain_output[:5])

Plain output shape: (32,)
First 5 outputs: [ 0.30577347  1.1445134  -0.22456057  0.60589397  0.54431903]


In [ ]:
encrypted_x = ts.ckks_vector(
    context,
    x.tolist()
)

print("Encrypted image created.")

Encrypted image created.


In [ ]:
encrypted_output = []

for j in range(weights1.shape[1]):

    encrypted_neuron = encrypted_x.dot(
        weights1[:, j].tolist()
    )

    encrypted_neuron = encrypted_neuron + float(bias1[j])

    encrypted_output.append(encrypted_neuron)

print("Encrypted Dense calculation completed.")

Encrypted Dense calculation completed.


In [ ]:
decrypted_output = np.array([
    encrypted_value.decrypt()[0]
    for encrypted_value in encrypted_output
])

print("Decrypted output shape:", decrypted_output.shape)
print("First 5 CKKS outputs:", decrypted_output[:5])

Decrypted output shape: (32,)
First 5 CKKS outputs: [ 0.3057888   1.14453104 -0.22454799  0.60591051  0.5443362 ]


In [ ]:
difference = np.abs(
    plain_output - decrypted_output
)

print("Plain output:")
print(plain_output[:5])

print("\nCKKS decrypted output:")
print(decrypted_output[:5])

print("\nMaximum difference:")
print(np.max(difference))

print("\nMean difference:")
print(np.mean(difference))

Plain output:
[ 0.30577347  1.1445134  -0.22456057  0.60589397  0.54431903]

CKKS decrypted output:
[ 0.3057888   1.14453104 -0.22454799  0.60591051  0.5443362 ]

Maximum difference:
2.1247300077131825e-05

Mean difference:
1.5117216101123146e-05


In [ ]:
encrypted_activation = []

for encrypted_value in encrypted_output:
    squared_value = encrypted_value * encrypted_value
    encrypted_activation.append(squared_value)

print("Encrypted polynomial activation completed.")

Encrypted polynomial activation completed.


In [ ]:
decrypted_activation = np.array([
    value.decrypt()[0]
    for value in encrypted_activation
])

print("First 5 decrypted activation values:")
print(decrypted_activation[:5])

First 5 decrypted activation values:
[0.09350708 1.30995548 0.05042196 0.36712871 0.29630285]


In [ ]:
plain_activation = plain_output ** 2

print("First 5 plaintext activation values:")
print(plain_activation[:5])

First 5 plaintext activation values:
[0.09349741 1.3099109  0.05042745 0.3671075  0.29628322]


In [ ]:
activation_difference = np.abs(
    plain_activation - decrypted_activation
)

print("Maximum activation difference:")
print(np.max(activation_difference))

print("\nMean activation difference:")
print(np.mean(activation_difference))

Maximum activation difference:
0.00013184955598966042

Mean activation difference:
3.658423842626983e-05


In [ ]:
dense2 = ckks_model.get_layer("dense_5")

weights2, bias2 = dense2.get_weights()

print("Weights shape:", weights2.shape)
print("Bias shape:", bias2.shape)

Weights shape: (32, 16)
Bias shape: (16,)


In [ ]:
plain_dense2 = plain_activation @ weights2 + bias2

print("Plain second Dense shape:", plain_dense2.shape)
print("First 5 plaintext outputs:", plain_dense2[:5])

Plain second Dense shape: (16,)
First 5 plaintext outputs: [-2.151195   0.6250746 -6.0737224  1.0201197  1.3420442]


In [ ]:
print(type(encrypted_activation))
print(len(encrypted_activation))
print(type(encrypted_activation[0]))

<class 'list'>
32
<class 'tenseal.tensors.ckksvector.CKKSVector'>


In [ ]:
# Second Dense layer: encrypted 32-value activation -> 16 outputs

# First create ONE CKKS vector containing all 32 encrypted activations
encrypted_activation_vector = ts.ckks_vector(
    context,
    plain_activation.tolist()
)

encrypted_dense2 = []

for j in range(weights2.shape[1]):

    # Dot product of 32 encrypted values with 32 plaintext weights
    encrypted_neuron = encrypted_activation_vector.dot(
        weights2[:, j].tolist()
    )

    # Add bias
    encrypted_neuron = encrypted_neuron + float(bias2[j])

    encrypted_dense2.append(encrypted_neuron)

print("Encrypted second Dense calculation completed.")

Encrypted second Dense calculation completed.


In [ ]:
encrypted_activation2 = []

for encrypted_value in encrypted_dense2:
    squared_value = encrypted_value * encrypted_value
    encrypted_activation2.append(squared_value)

print("Number of encrypted activation values:", len(encrypted_activation2))

Number of encrypted activation values: 16


In [ ]:
decrypted_activation2 = np.array([
    value.decrypt()[0]
    for value in encrypted_activation2
])

print("Decrypted activation shape:", decrypted_activation2.shape)
print("First 5 values:", decrypted_activation2[:5])

Decrypted activation shape: (16,)
First 5 values: [ 4.6276628   0.3907226  36.89030215  1.04065362  1.80109756]


In [ ]:
plain_activation2 = plain_dense2 ** 2

print("Plain activation shape:", plain_activation2.shape)
print("First 5 values:", plain_activation2[:5])

Plain activation shape: (16,)
First 5 values: [ 4.6276402   0.39071828 36.890102    1.0406442   1.8010827 ]


In [ ]:
activation2_difference = np.abs(
    plain_activation2 - decrypted_activation2
)

print("Plaintext:")
print(plain_activation2[:5])

print("\nCKKS decrypted:")
print(decrypted_activation2[:5])

print("\nMaximum difference:")
print(np.max(activation2_difference))

print("\nMean difference:")
print(np.mean(activation2_difference))

Plaintext:
[ 4.6276402   0.39071828 36.890102    1.0406442   1.8010827 ]

CKKS decrypted:
[ 4.6276628   0.3907226  36.89030215  1.04065362  1.80109756]

Maximum difference:
0.0004767594433872091

Mean difference:
6.250581865037435e-05


In [ ]:
dense3 = ckks_model.get_layer("dense_6")

weights3, bias3 = dense3.get_weights()

print("Weights shape:", weights3.shape)
print("Bias shape:", bias3.shape)

Weights shape: (16, 1)
Bias shape: (1,)


In [ ]:
plain_final = plain_activation2 @ weights3 + bias3

print("Plain final output:")
print(plain_final)

Plain final output:
[-5.096177]


In [ ]:
print("Number of encrypted second activations:", len(encrypted_activation2))
print("Final weights shape:", weights3.shape)

Number of encrypted second activations: 16
Final weights shape: (16, 1)


In [ ]:
print("weights3 shape:", weights3.shape)
print("bias3 shape:", bias3.shape)
print("plain_activation2 shape:", plain_activation2.shape)

weights3 shape: (16, 1)
bias3 shape: (1,)
plain_activation2 shape: (16,)


In [ ]:
encrypted_activation_vector1 = ts.ckks_vector(
    context,
    plain_activation2.tolist()
)

encrypted_final = []

for j in range(weights3.shape[1]):

    # Dot product of 32 encrypted values with 32 plaintext weights
    encrypted_neuron = encrypted_activation_vector1.dot(
        weights3[:, j].tolist()
    )

    # Add bias
    encrypted_neuron = encrypted_neuron + float(bias3[j])

    encrypted_final.append(encrypted_neuron)

In [ ]:
decrypted_final = np.array([
    value.decrypt()[0]
    for value in encrypted_final
])

print("Decrypted activation shape:", decrypted_final.shape)
print("First 5 values:", decrypted_final[:5])

Decrypted activation shape: (1,)
First 5 values: [-5.0961826]


In [ ]:
final_difference = np.abs(
    plain_final - decrypted_final
)

print("Plaintext:")
print(plain_final)

print("\nCKKS decrypted:")
print(decrypted_final)

print("\ndifference:")
print(final_difference)

Plaintext:
[-5.096177]

CKKS decrypted:
[-5.0961826]

difference:
[5.50174011e-06]


In [ ]:
prediction = 1 if decrypted_final >= 0 else 0

print("Predicted class:", prediction)
print("True class:", true_label)

Predicted class: 0
True class: 0.0
